In [12]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 01 — RAW DATA PROFILING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

# Notebook is located inside the /notebooks folder
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CREDRESOLVE — RAW DATA PROFILING")
print("=" * 80)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data    : {RAW_DIR}")
print(f"Output      : {OUTPUT_DIR}")

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR}")


# ------------------------------------------------------------
# 2. DISCOVER RAW CSV FILES
# ------------------------------------------------------------

csv_files = sorted(RAW_DIR.glob("*.csv"))

print(f"\nCSV files found: {len(csv_files)}")

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in data/raw.")

for file in csv_files:
    print(f"  - {file.name}")


# ------------------------------------------------------------
# 3. LOAD DATASETS
# ------------------------------------------------------------

datasets = {}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

print("\nDatasets loaded successfully.")


# ------------------------------------------------------------
# 4. DATASET-LEVEL INVENTORY
# ------------------------------------------------------------

inventory = []

for name, df in datasets.items():

    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())

    inventory.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": missing_cells,
        "missing_pct_all_cells": round(
            (missing_cells / total_cells) * 100, 2
        ) if total_cells else 0
    })

inventory_df = (
    pd.DataFrame(inventory)
    .sort_values("dataset")
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("DATASET INVENTORY")
print("=" * 80)

display(inventory_df)

inventory_df.to_csv(
    OUTPUT_DIR / "dataset_inventory.csv",
    index=False
)


# ------------------------------------------------------------
# 5. COLUMN-LEVEL PROFILE
# ------------------------------------------------------------

column_profiles = []

for name, df in datasets.items():

    for column in df.columns:

        series = df[column]

        column_profiles.append({
            "dataset": name,
            "column": column,
            "data_type": str(series.dtype),
            "row_count": len(df),
            "missing_count": int(series.isna().sum()),
            "missing_pct": round(
                series.isna().mean() * 100, 2
            ),
            "unique_values": int(
                series.nunique(dropna=True)
            )
        })

column_profile_df = pd.DataFrame(column_profiles)

print("\n" + "=" * 80)
print("COLUMN-LEVEL PROFILE")
print("=" * 80)

display(column_profile_df.head(30))

column_profile_df.to_csv(
    OUTPUT_DIR / "column_profile.csv",
    index=False
)


# ------------------------------------------------------------
# 6. MISSING-VALUE PROFILE
# ------------------------------------------------------------

missing_profile = (
    column_profile_df[
        column_profile_df["missing_count"] > 0
    ]
    .sort_values(
        ["missing_pct", "dataset"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MISSING-VALUE PROFILE")
print("=" * 80)

display(missing_profile)

missing_profile.to_csv(
    OUTPUT_DIR / "missing_value_profile.csv",
    index=False
)


# ------------------------------------------------------------
# 7. DUPLICATE-ROW PROFILE
# ------------------------------------------------------------

duplicate_profile = (
    inventory_df[
        inventory_df["duplicate_rows"] > 0
    ]
    .sort_values(
        "duplicate_rows",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("DUPLICATE-ROW PROFILE")
print("=" * 80)

display(duplicate_profile)

duplicate_profile.to_csv(
    OUTPUT_DIR / "duplicate_row_profile.csv",
    index=False
)


# ------------------------------------------------------------
# 8. CANDIDATE KEY DISCOVERY
# ------------------------------------------------------------
# IMPORTANT:
# These are only candidates.
# We will NOT declare them as primary keys yet.
#
# A candidate key must:
#   - contain no missing values
#   - contain unique values across the table
# ------------------------------------------------------------

candidate_keys = []

for name, df in datasets.items():

    for column in df.columns:

        series = df[column]

        if (
            series.notna().all()
            and series.nunique(dropna=True) == len(df)
        ):
            candidate_keys.append({
                "dataset": name,
                "column": column,
                "rows": len(df),
                "unique_values": int(
                    series.nunique(dropna=True)
                )
            })

candidate_keys_df = pd.DataFrame(candidate_keys)

print("\n" + "=" * 80)
print("CANDIDATE KEY PROFILE")
print("=" * 80)

display(candidate_keys_df)

candidate_keys_df.to_csv(
    OUTPUT_DIR / "candidate_keys.csv",
    index=False
)


# ------------------------------------------------------------
# 9. IDENTIFIER COLUMN DISCOVERY
# ------------------------------------------------------------

identifier_keywords = [
    "id",
    "reference",
    "ref",
    "code"
]

identifier_profile = []

for name, df in datasets.items():

    for column in df.columns:

        column_lower = column.lower()

        if any(
            keyword in column_lower
            for keyword in identifier_keywords
        ):

            identifier_profile.append({
                "dataset": name,
                "column": column,
                "data_type": str(df[column].dtype),
                "rows": len(df),
                "unique_values": int(
                    df[column].nunique(dropna=True)
                ),
                "missing_count": int(
                    df[column].isna().sum()
                ),
                "unique_pct": round(
                    df[column].nunique(dropna=True)
                    / len(df) * 100,
                    2
                )
            })

identifier_profile_df = pd.DataFrame(
    identifier_profile
)

print("\n" + "=" * 80)
print("IDENTIFIER PROFILE")
print("=" * 80)

display(identifier_profile_df)

identifier_profile_df.to_csv(
    OUTPUT_DIR / "identifier_profile.csv",
    index=False
)


# ------------------------------------------------------------
# 10. DATE / TIMESTAMP COLUMN DISCOVERY
# ------------------------------------------------------------

date_keywords = [
    "date",
    "time",
    "timestamp",
    "created",
    "updated",
    "scheduled",
    "started",
    "ended",
    "occurred",
    "paid",
    "opened",
    "closed",
    "effective"
]

date_columns = []

for name, df in datasets.items():

    for column in df.columns:

        column_lower = column.lower()

        if any(
            keyword in column_lower
            for keyword in date_keywords
        ):

            date_columns.append({
                "dataset": name,
                "column": column,
                "data_type": str(df[column].dtype),
                "missing_count": int(
                    df[column].isna().sum()
                )
            })

date_columns_df = pd.DataFrame(date_columns)

print("\n" + "=" * 80)
print("DATE / TIMESTAMP COLUMNS")
print("=" * 80)

display(date_columns_df)

date_columns_df.to_csv(
    OUTPUT_DIR / "date_timestamp_columns.csv",
    index=False
)


# ------------------------------------------------------------
# 11. DATE RANGE PROFILE
# ------------------------------------------------------------
# Parsing is performed on temporary analysis objects.
# RAW CSV files are NOT modified.
# ------------------------------------------------------------

date_ranges = []

for name, df in datasets.items():

    for column in df.columns:

        column_lower = column.lower()

        if any(
            keyword in column_lower
            for keyword in date_keywords
        ):

            parsed = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            valid_dates = parsed.dropna()

            if len(valid_dates) > 0:

                date_ranges.append({
                    "dataset": name,
                    "column": column,
                    "valid_dates": len(valid_dates),
                    "unparsed_values": int(
                        df[column].notna().sum()
                        - len(valid_dates)
                    ),
                    "min_date": valid_dates.min(),
                    "max_date": valid_dates.max()
                })

date_ranges_df = pd.DataFrame(date_ranges)

print("\n" + "=" * 80)
print("DATE RANGE PROFILE")
print("=" * 80)

display(date_ranges_df)

date_ranges_df.to_csv(
    OUTPUT_DIR / "date_range_profile.csv",
    index=False
)


# ------------------------------------------------------------
# 12. OFFICIAL DATA DICTIONARY
# ------------------------------------------------------------

dictionary_path = RAW_DIR / "data_dictionary.csv"

if dictionary_path.exists():

    dictionary_df = pd.read_csv(dictionary_path)

    print("\n" + "=" * 80)
    print("DATA DICTIONARY")
    print("=" * 80)

    print(
        f"Dictionary entries: {len(dictionary_df)}"
    )

    print(
        f"Dictionary columns: "
        f"{dictionary_df.columns.tolist()}"
    )

    display(dictionary_df.head(30))

else:
    dictionary_df = None
    print("\nData dictionary not found.")


# ------------------------------------------------------------
# 13. PROFILING SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame([{
    "datasets_loaded": len(datasets),
    "total_rows_across_datasets": sum(
        len(df) for df in datasets.values()
    ),
    "total_columns_across_datasets": sum(
        len(df.columns) for df in datasets.values()
    ),
    "datasets_with_duplicates": int(
        (inventory_df["duplicate_rows"] > 0).sum()
    ),
    "datasets_with_missing_values": int(
        (inventory_df["missing_cells"] > 0).sum()
    ),
    "candidate_key_columns": len(
        candidate_keys_df
    ),
    "date_timestamp_columns": len(
        date_columns_df
    )
}])

print("\n" + "=" * 80)
print("PROFILING SUMMARY")
print("=" * 80)

display(summary)

summary.to_csv(
    OUTPUT_DIR / "profiling_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RAW DATA PROFILING COMPLETE")
print("=" * 80)

print(f"Datasets analyzed : {len(datasets)}")
print(f"Profiling outputs : {OUTPUT_DIR}")
print("Raw source files  : NOT MODIFIED")

CREDRESOLVE — RAW DATA PROFILING
Project root : c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics
Raw data    : c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\data\raw
Output      : c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables

CSV files found: 18
  - account_status_history.csv
  - accounts.csv
  - agent_sessions.csv
  - agents.csv
  - borrowers.csv
  - call_attempts.csv
  - call_dispositions.csv
  - calls.csv
  - campaigns.csv
  - complaints.csv
  - daily_targeting.csv
  - data_dictionary.csv
  - field_visits.csv
  - payments.csv
  - promises_to_pay.csv
  - sms_events.csv
  - vendor_telephony.csv
  - whatsapp_events.csv

Datasets loaded successfully.

DATASET INVENTORY


,dataset,rows,columns,duplicate_rows,missing_cells,missing_pct_all_cells
0,account_status_history,60000,8,0,0,0.00
1,accounts,30000,11,0,455,0.14
2,agent_sessions,15000,7,0,0,0.00
3,agents,30000,8,0,0,0.00
4,borrowers,30600,8,600,1509,0.62
5,call_attempts,120000,9,0,2400,0.22
6,call_dispositions,35000,8,0,0,0.00
7,calls,91350,11,1271,1827,0.18
8,campaigns,120,7,0,0,0.00
9,complaints,8000,9,0,0,0.00



COLUMN-LEVEL PROFILE


,dataset,column,data_type,row_count,missing_count,missing_pct,unique_values
0,account_status_history,history_id,object,60000,0,0.00,60000
1,account_status_history,account_id,object,60000,0,0.00,25999
2,account_status_history,borrower_id,object,60000,0,0.00,11916
3,account_status_history,event_at,object,60000,0,0.00,59898
4,account_status_history,status,object,60000,0,0.00,7
5,account_status_history,changed_by,object,60000,0,0.00,101
6,account_status_history,source,object,60000,0,0.00,5
7,account_status_history,recorded_at,object,60000,0,0.00,59906
8,accounts,account_id,object,30000,0,0.00,30000
9,accounts,borrower_id,object,30000,455,1.52,10943



MISSING-VALUE PROFILE


...


DATA DICTIONARY
Dictionary entries: 143
Dictionary columns: ['dataset', 'column', 'dtype']


,dataset,column,dtype
0,borrowers,borrower_id,object
1,borrowers,name,object
2,borrowers,phone,object
3,borrowers,email,object
4,borrowers,city,object
5,borrowers,created_at,datetime64[ns]
6,borrowers,updated_at,datetime64[ns]
7,borrowers,state,object
8,accounts,account_id,object
9,accounts,borrower_id,object



PROFILING SUMMARY


,datasets_loaded,total_rows_across_datasets,total_columns_across_datasets,datasets_with_duplicates,datasets_with_missing_values,candidate_key_columns,date_timestamp_columns
0,18,639328,146,4,6,17,12



RAW DATA PROFILING COMPLETE
Datasets analyzed : 18
Profiling outputs : c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
Raw source files  : NOT MODIFIED
